# 1. IMPORTS

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import r2_score, mean_squared_error

# 2. LOAD DATA 

In [2]:
df = pd.read_csv("data.csv")

print("Original Shape:", df.shape)

Original Shape: (4600, 18)


# 3. DATA CLEANING 

In [3]:
# Drop high-cardinality / unnecessary columns
cols_to_drop = ['date', 'country', 'statezip', 'street']
df = df.drop(columns=[col for col in cols_to_drop if col in df.columns])

# Remove extreme outliers (top 1%)
upper_limit = df["price"].quantile(0.99)
df = df[df["price"] <= upper_limit]

print("Shape After Cleaning:", df.shape)

Shape After Cleaning: (4554, 14)


# 4. SPLIT FEATURES & TARGET

In [4]:
X = df.drop("price", axis=1)
y = df["price"]

# 5. TRAIN-TEST SPLIT (NO LEAKAGE)

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# 6. IDENTIFY FEATURE TYPES

In [6]:
numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X_train.select_dtypes(include=['object']).columns

# 7. PREPROCESSING PIPELINE

In [7]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

# 8. LINEAR REGRESSION

In [8]:
linear_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

linear_model.fit(X_train, y_train)

lin_pred = linear_model.predict(X_test)

lin_r2 = r2_score(y_test, lin_pred)
lin_rmse = np.sqrt(mean_squared_error(y_test, lin_pred))

print("\n----- Linear Regression -----")
print("R2 Score:", lin_r2)
print("RMSE:", lin_rmse)


----- Linear Regression -----
R2 Score: 0.6535748987162497
RMSE: 171098.17040262284


# 9. RIDGE REGRESSION + TUNING

In [9]:
best_r2 = -999
best_alpha = None
best_model = None

for alpha in [0.01, 0.1, 1, 10, 50, 100]:
    
    ridge_model = Pipeline([
        ("preprocessor", preprocessor),
        ("model", Ridge(alpha=alpha))
    ])
    
    ridge_model.fit(X_train, y_train)
    pred = ridge_model.predict(X_test)
    score = r2_score(y_test, pred)
    
    print("Alpha:", alpha, "R2:", score)
    
    if score > best_r2:
        best_r2 = score
        best_alpha = alpha
        best_model = ridge_model

print("\nBest Alpha:", best_alpha)
print("Best Ridge R2:", best_r2)

ridge_rmse = np.sqrt(mean_squared_error(y_test, best_model.predict(X_test)))
print("Best Ridge RMSE:", ridge_rmse)

Alpha: 0.01 R2: 0.653457297419313
Alpha: 0.1 R2: 0.653381967558071
Alpha: 1 R2: 0.6525187196372296
Alpha: 10 R2: 0.6458640691588309
Alpha: 50 R2: 0.6287879085983215
Alpha: 100 R2: 0.6120829884728891

Best Alpha: 0.01
Best Ridge R2: 0.653457297419313
Best Ridge RMSE: 171127.2093771409


# 10. PREDICTION ON UNSEEN DATA

In [10]:
new_house = pd.DataFrame({
    "bedrooms": [3],
    "bathrooms": [2],
    "sqft_living": [1800],
    "sqft_lot": [4000],
    "floors": [1],
    "waterfront": [0],
    "view": [0],
    "condition": [3],
    "sqft_above": [1500],
    "sqft_basement": [300],
    "yr_built": [2005],
    "yr_renovated": [0],
    "city": ["Seattle"]
})

predicted_price = best_model.predict(new_house)

print("\n----- Unseen House Prediction -----")
print("Predicted Price:", predicted_price[0])


----- Unseen House Prediction -----
Predicted Price: 455512.0506398607
